In [2]:
# ==========================================
# AUDIO STRESS MODEL (ZIP UPLOAD) - FROM SCRATCH
# ==========================================
# You will upload one or more ZIP files (RAVDESS-like).
# The code:
#   1) Upload ZIP(s)
#   2) Extract all .wav into /content/audio
#   3) Build 3-class labels from emotion code in filename
#   4) Train LogisticRegression model
#   5) Save .joblib + meta json
# ==========================================

# ---------- 0) Imports ----------
import os, glob, zipfile, shutil, json, random
import numpy as np
import librosa

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

In [3]:
# ---------- 1) Upload ZIP file(s) manually ----------
# Upload one ZIP, or multiple ZIPs (e.g., Actor_01.zip, Actor_02.zip...)

uploaded = files.upload()

zip_paths = []
for name in uploaded.keys():
    if name.lower().endswith(".zip"):
        zip_paths.append("/content/" + name)

print("Uploaded ZIPs:", zip_paths)
assert len(zip_paths) > 0, "No ZIP file uploaded."

Saving Audio_Speech_Actors_01-24.zip to Audio_Speech_Actors_01-24.zip
Uploaded ZIPs: ['/content/Audio_Speech_Actors_01-24.zip']


In [4]:
# ---------- 2) Extract ZIP(s) into /content/audio ----------
AUDIO_DIR = "/content/audio"
os.makedirs(AUDIO_DIR, exist_ok=True)

def extract_zip_to_audio(zip_path: str, out_dir: str):
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("/content/_tmp_extract")

    # move .wav files into out_dir
    count = 0
    for root, _, files_list in os.walk("/content/_tmp_extract"):
        for f in files_list:
            if f.lower().endswith(".wav"):
                src = os.path.join(root, f)
                dst = os.path.join(out_dir, f)
                # avoid overwrite
                if os.path.exists(dst):
                    base, ext = os.path.splitext(f)
                    dst = os.path.join(out_dir, base + "_" + str(count) + ext)
                shutil.move(src, dst)
                count += 1

    shutil.rmtree("/content/_tmp_extract", ignore_errors=True)
    return count

total = 0
for zp in zip_paths:
    c = extract_zip_to_audio(zp, AUDIO_DIR)
    print("Extracted", c, "wav files from", os.path.basename(zp))
    total += c

print("\nTotal wav files in /content/audio:", len(glob.glob(AUDIO_DIR + "/*.wav")))
assert total > 0, "No wav files extracted. Check ZIP contents."

Extracted 1440 wav files from Audio_Speech_Actors_01-24.zip

Total wav files in /content/audio: 1440


In [5]:
# ---------- 2) Extract ZIP(s) into /content/audio ----------
AUDIO_DIR = "/content/audio"
os.makedirs(AUDIO_DIR, exist_ok=True)

def extract_zip_to_audio(zip_path: str, out_dir: str):
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("/content/_tmp_extract")

    # move .wav files into out_dir
    count = 0
    for root, _, files_list in os.walk("/content/_tmp_extract"):
        for f in files_list:
            if f.lower().endswith(".wav"):
                src = os.path.join(root, f)
                dst = os.path.join(out_dir, f)
                # avoid overwrite
                if os.path.exists(dst):
                    base, ext = os.path.splitext(f)
                    dst = os.path.join(out_dir, base + "_" + str(count) + ext)
                shutil.move(src, dst)
                count += 1

    shutil.rmtree("/content/_tmp_extract", ignore_errors=True)
    return count

total = 0
for zp in zip_paths:
    c = extract_zip_to_audio(zp, AUDIO_DIR)
    print("Extracted", c, "wav files from", os.path.basename(zp))
    total += c

print("\nTotal wav files in /content/audio:", len(glob.glob(AUDIO_DIR + "/*.wav")))
assert total > 0, "No wav files extracted. Check ZIP contents."

Extracted 1440 wav files from Audio_Speech_Actors_01-24.zip

Total wav files in /content/audio: 2880


In [6]:
# ---------- 3) Labeling: emotion code -> stress level ----------
# RAVDESS filename format (common):
#   03-01-EMO-... .wav
# where EMO = parts[2]
#
# Stress mapping (editable):
#   LOW      : 01,02
#   MODERATE : 03,04
#   HIGH     : 05,06,07,08

LOW_EMO  = {"01", "02"}
MOD_EMO  = {"03", "04"}
HIGH_EMO = {"05", "06", "07", "08"}

def emo_code_from_name(path: str):
    base = os.path.basename(path)
    parts = base.split("-")
    if len(parts) < 3:
        return None
    return parts[2]

def stress_label_from_emo(emo: str):
    if emo in LOW_EMO:
        return "Low"
    if emo in MOD_EMO:
        return "Moderate"
    if emo in HIGH_EMO:
        return "High"
    return None

In [7]:
# ---------- 4) Feature extractor (94 features, consistent) ----------
# 1 sec fixed window + MFCC mean/std + chroma mean + zcr + rms
# total: 40+40+12+2 = 94

def extract_audio_features(path: str, sr: int = 16000) -> np.ndarray:
    y, sr = librosa.load(path, sr=sr)
    y, _ = librosa.effects.trim(y, top_db=25)

    target_len = int(sr * 1.0)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    mfcc_mean = mfcc.mean(axis=1)
    mfcc_std  = mfcc.std(axis=1)

    chroma = librosa.feature.chroma_stft(y=y, sr=sr).mean(axis=1)

    zcr = float(librosa.feature.zero_crossing_rate(y).mean())
    rms = float(librosa.feature.rms(y=y).mean())

    feat = np.hstack([mfcc_mean, mfcc_std, chroma, [zcr, rms]]).astype(np.float32)
    return feat

In [8]:
# ---------- 5) Build dataset X, y ----------
audio_files = sorted(glob.glob(AUDIO_DIR + "/*.wav"))

X, y = [], []
skipped = 0
emo_counts = {}
label_counts = {"Low": 0, "Moderate": 0, "High": 0}

for f in audio_files:
    emo = emo_code_from_name(f)
    emo_counts[emo] = emo_counts.get(emo, 0) + 1
    lab = stress_label_from_emo(emo) if emo else None
    if lab is None:
        skipped += 1
        continue
    feat = extract_audio_features(f)
    X.append(feat)
    y.append(lab)
    label_counts[lab] += 1

X = np.vstack(X)
y = np.array(y)

print("X shape:", X.shape, "(must be Nx94)")
print("Label counts:", label_counts)
print("Skipped files:", skipped)
print("Emotion code counts:", emo_counts)

assert X.shape[1] == 94, f"Feature dimension mismatch: got {X.shape[1]}, expected 94"
assert len(np.unique(y)) >= 2, "Not enough classes found. Check ZIP contents and filename format."

/usr/local/lib/python3.12/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


X shape: (2880, 94) (must be Nx94)
Label counts: {'Low': 576, 'Moderate': 768, 'High': 1536}
Skipped files: 0
Emotion code counts: {'01': 192, '02': 384, '03': 384, '04': 384, '05': 384, '06': 384, '07': 384, '08': 384}


In [9]:
# ---------- 6) Train-test split ----------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# ---------- 7) Train model (3-class Logistic Regression) ----------
audio_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        multi_class="auto"
    ))
])

audio_pipe.fit(X_train, y_train)

pred = audio_pipe.predict(X_test)
acc = accuracy_score(y_test, pred)

print("\nAccuracy:", round(acc, 4))
print("\nConfusion matrix [Low, Moderate, High]:")
print(confusion_matrix(y_test, pred, labels=["Low", "Moderate", "High"]))
print("\nClassification report:")
print(classification_report(y_test, pred))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



Accuracy: 0.7069

Confusion matrix [Low, Moderate, High]:
[[123  13   8]
 [ 39 105  48]
 [ 19  84 281]]

Classification report:
              precision    recall  f1-score   support

        High       0.83      0.73      0.78       384
         Low       0.68      0.85      0.76       144
    Moderate       0.52      0.55      0.53       192

    accuracy                           0.71       720
   macro avg       0.68      0.71      0.69       720
weighted avg       0.72      0.71      0.71       720



In [10]:
# ---------- 8) Save model + metadata ----------
OUT_MODEL = "/content/stress_audio_pipeline_3class.joblib"
OUT_META  = "/content/stress_audio_pipeline_3class_meta.json"

joblib.dump(audio_pipe, OUT_MODEL)

meta = {
    "sr": 16000,
    "window_sec": 1.0,
    "features": "94 (mfcc mean40 + mfcc std40 + chroma mean12 + zcr1 + rms1)",
    "label_map": {
        "Low": sorted(list(LOW_EMO)),
        "Moderate": sorted(list(MOD_EMO)),
        "High": sorted(list(HIGH_EMO)),
    },
    "classes_": list(audio_pipe.named_steps["clf"].classes_),
    "label_counts": label_counts
}

with open(OUT_META, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved model:", OUT_MODEL)
print("Saved meta :", OUT_META)

# download to your laptop
files.download(OUT_MODEL)
files.download(OUT_META)

Saved model: /content/stress_audio_pipeline_3class.joblib
Saved meta : /content/stress_audio_pipeline_3class_meta.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
# ---------- 9) Manual test: pick any extracted wav and predict ----------
def audio_predict_one(wav_path: str):
    feat = extract_audio_features(wav_path).reshape(1, -1)
    proba = audio_pipe.predict_proba(feat)[0]
    classes = audio_pipe.named_steps["clf"].classes_
    best = classes[int(np.argmax(proba))]
    return best, {c: float(p) for c, p in zip(classes, proba)}

sample = random.choice(sorted(glob.glob("/content/audio/*.wav")))
pred_label, pred_proba = audio_predict_one(sample)

print("Sample file:", os.path.basename(sample))
print("Pred:", pred_label)
print("Proba:", pred_proba)

Sample file: 03-01-03-02-01-02-01.wav
Pred: Moderate
Proba: {np.str_('High'): 0.31459145925452003, np.str_('Low'): 0.0015767154380988524, np.str_('Moderate'): 0.6838318253073811}


In [12]:
# =====================================
# MANUAL AUDIO TEST (UPLOAD A FILE)
# =====================================
from google.colab import files
import numpy as np

uploaded = files.upload()   # choose any .wav file

for fname in uploaded.keys():

    # extract features
    feat = extract_audio_features(fname).reshape(1, -1)

    # predict
    pred = audio_pipe.predict(feat)[0]
    proba = audio_pipe.predict_proba(feat)[0]

    classes = audio_pipe.named_steps["clf"].classes_

    print("\nFile:", fname)
    print("Predicted Stress Level:", pred)

    print("\nClass Probabilities:")
    for c, p in zip(classes, proba):
        print(c, ":", round(float(p), 3))

Saving 03-01-01-01-01-01-04.wav to 03-01-01-01-01-01-04.wav
Saving 03-01-01-01-01-02-04.wav to 03-01-01-01-01-02-04.wav
Saving 03-01-01-01-02-01-04.wav to 03-01-01-01-02-01-04.wav

File: 03-01-01-01-01-01-04.wav
Predicted Stress Level: High

Class Probabilities:
High : 0.448
Low : 0.295
Moderate : 0.257

File: 03-01-01-01-01-02-04.wav
Predicted Stress Level: Low

Class Probabilities:
High : 0.009
Low : 0.981
Moderate : 0.01

File: 03-01-01-01-02-01-04.wav
Predicted Stress Level: Low

Class Probabilities:
High : 0.044
Low : 0.86
Moderate : 0.096


In [14]:
# pick any audio file from extracted dataset
import random, glob

f = random.choice(glob.glob("/content/audio/*.wav"))

feat = extract_audio_features(f).reshape(1,-1)

pred = audio_pipe.predict(feat)[0]
proba = audio_pipe.predict_proba(feat)[0]

print("File:", f)
print("Stress Level:", pred)

File: /content/audio/03-01-05-01-01-01-16_876.wav
Stress Level: High
